# Synthetic QCN Backbone — Demo

**Controlled, globally-consistent Qualitative Constraint Networks (QCNs)** over three
relation algebras, with exact gold relations and natural-language realizations.

This is the *clean-ground-truth backbone* for the **redundancy** and **iteration**
claims of the *No Derivation, No Relation* closure-certificate work: a controlled QCN
sweep **cannot be found, only generated** — no real corpus independently exposes
redundancy, hop-count, cyclomatic number, density and node-count with exact, consistent
gold.

Each network is a realizable **geometric model**:

| algebra | role | base relations | model |
|---|---|---|---|
| `point` (convex Point) | primary | `< = >` | integer coordinates; gold = `sign(x_i − x_j)` |
| `allen` (Allen Interval) | primary | 13 relations `b bi m mi o oi d di s si f fi eq` | integer-grid intervals `(s,e)` |
| `rcc8` (Region Connection Calculus) | secondary | 8 relations `DC EC PO EQ TPP NTPP TPPi NTPPi` | collinear integer discs |

Because every network is a realizable model, it is **globally consistent by
construction** (no solver needed) and the gold atomic relation on each edge is read
*exactly* off the model. Each network has a **held-out query pair** `(s,t)` that shares
no edge and never co-occurs in one sentence — the query relation is obtainable only by
**composing ≥1 multi-edge path** (deduction-required).

**What this demo does:** it reproduces the generator at small scale (a few networks per
structural cell), runs the *correctness gate* (composition of gold relations along every
enumerated `s→t` path must contain the gold query relation), and reproduces the headline
structural signal — *singleton-resolution rises monotonically with redundancy `P`*. It
also loads the published curated subset from GitHub to show the canonical output format.

> The code below is the original `method.py` + `qcn/` package, split into cells with
> explanations. The only changes from the original: algebra tables are embedded instead
> of read from disk, the multiprocessing pool is replaced by a serial loop, and scale is
> controlled by the config cell.

## Setup

### Install dependencies

The generator needs only `numpy`, `networkx` and `loguru` (plus `matplotlib` for the
plots at the end). On Colab the core scientific packages are pre-installed — we only
install them locally, pinned to Colab's versions, so the environments match.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab -> always install
_pip('loguru==0.7.3')

# numpy / networkx / matplotlib ARE pre-installed on Colab -> install locally only,
# at Colab's exact versions (installing them ON Colab corrupts the loaded C extensions).
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'networkx==3.6.1', 'matplotlib==3.10.0')

### Imports

Copied from the original `method.py` import block (CLI/path-only imports dropped), plus
`matplotlib` for the visualization cell.

In [ ]:
import hashlib
import json
import sys
from collections import Counter, defaultdict
from itertools import islice
import types

import networkx as nx
import numpy as np
from loguru import logger
import matplotlib.pyplot as plt

# keep loguru output compact (used by the worker bridge _job on failures)
logger.remove()
logger.add(sys.stderr, level="INFO", format="{level:<7}|{message}")

### Load the published curated subset (from GitHub, with local fallback)

`mini_demo_data.json` is a curated subset of the *published* corpus: one example per
structural cell from the `synthetic_qcn_allen` dataset. We use it to show the canonical
**output schema** and to compare the demo-generated structural signal against the full
corpus.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-1/dataset-2/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("top-level keys:", list(data.keys()))
print("datasets:", [d["dataset"] for d in data["datasets"]])
print("published examples in subset:", sum(len(d["examples"]) for d in data["datasets"]))
print("cells covered:", [e["metadata_cell"]["cell_id"] for e in data["datasets"][0]["examples"]])

## Config

All tunable knobs live here. The **scale knobs** start at the absolute minimum (1 network
per structural cell). Increase them to make the structural-signal estimates smoother — the
full published corpus uses **500/cell** (primary) and **300/cell** (secondary). The
remaining values are correctness/structural constants copied verbatim from `method.py`
and are *not* scale knobs.

In [ ]:
# ---- DEMO SCALE KNOBS (smallest values that produce output; scale up as desired) ----
NETWORKS_PER_CELL_PRIMARY   = 40    # point & allen networks per structural cell. FULL: 500
NETWORKS_PER_CELL_SECONDARY = 25    # rcc8 networks per structural cell.          FULL: 300

# ---- correctness / structural config (verbatim from method.py; NOT scale knobs) ----
PATH_CAP = 64                 # max enumerated s-t paths per network
LEN_CUTOFF = 8                # max simple-path length explored
PRIMARY = ("point", "allen")  # full grid, primary algebras
SECONDARY = ("rcc8",)         # secondary algebra
REF_LABEL_L = {"point": None, "allen": 6.5, "rcc8": 4.0}  # avg disjunctive-label size

REALISM_PREREG = {
    "validated": False,
    "note": ("Pre-registered realism-matching thresholds, fixed now to inoculate "
             "against post-hoc tuning. To be checked NEXT iteration against the real "
             "frontier-pilot (NarrativeTime/TDDMan/MATRES) error distributions."),
    "tv_distance_max": 0.15,          # per-edge error-type distribution (TV distance)
    "rho_match_tol": 0.10,            # cross-edge error-correlation rho match tolerance
    "topology_histogram_emd_max": 0.10,  # contributing-edge-count + cycle-structure EMD
}

## The relation algebras (`qcn/algebras.py`)

### Composition tables (embedded)

Composition (`TransTable`) and converse maps come from the **authoritative
`alreich/qualreas`** algebra definitions (Allen-1983 interval composition; Renz & Nebel
RCC-8 composition; linear point algebra). In the original package these are read from
`qcn/algebra_tables/*.json`; here we embed the exact same JSON so the notebook is
self-contained.

In [ ]:
ALGEBRA_TABLE_JSON = {
    "Linear_Point_Algebra.json": json.loads(r'''{
    "Description": "Linear Point Algebra", 
    "Name": "Linear_Point_Algebra",
    "Relations": {
        "<": {
            "Domain": ["Point"], 
            "Converse": ">", 
            "Name": "LessThan", 
            "Range": ["Point"], 
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": true
        }, 
        "=": {
            "Domain": ["Point"], 
            "Converse": "=", 
            "Name": "Equals", 
            "Range": ["Point"], 
            "Reflexive": true, 
            "Symmetric": true, 
            "Transitive": true
        }, 
        ">": {
            "Domain": ["Point"], 
            "Converse": "<", 
            "Name": "GreaterThan", 
            "Range": ["Point"], 
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": true
        }
    }, 
    "TransTable": {
        "<": {
            "<": "<",
            "=": "<",
            ">": "<|=|>"
        },
        "=": {
            "<": "<",
            "=": "=",
            ">": ">"
        },
        ">": {
            "<": "<|=|>",
            "=": ">",
            ">": ">"
        }
    }
}
'''),
    "Linear_Interval_Algebra.json": json.loads(r'''{
    "Description": "Allen's algebra of proper time intervals",
    "Name": "Linear_Interval_Algebra",
    "Relations": {
        "B": {
            "Domain": ["ProperInterval"],
            "Converse": "BI",
            "Name": "Before",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "BI": {
            "Domain": ["ProperInterval"],
            "Converse": "B",
            "Name": "After",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "D": {
            "Domain": ["ProperInterval"],
            "Converse": "DI",
            "Name": "During",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "DI": {
            "Domain": ["ProperInterval"],
            "Converse": "D",
            "Name": "Contains",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "E": {
            "Domain": ["ProperInterval"],
            "Converse": "E",
            "Name": "Equals",
            "Range": ["ProperInterval"],
            "Reflexive": true,
            "Symmetric": true,
            "Transitive": true
        },
        "F": {
            "Domain": ["ProperInterval"],
            "Converse": "FI",
            "Name": "Finishes",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "FI": {
            "Domain": ["ProperInterval"],
            "Converse": "F",
            "Name": "Finished-by",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "M": {
            "Domain": ["ProperInterval"],
            "Converse": "MI",
            "Name": "Meets",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": false
        },
        "MI": {
            "Domain": ["ProperInterval"],
            "Converse": "M",
            "Name": "Met-By",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": false
        },
        "O": {
            "Domain": ["ProperInterval"],
            "Converse": "OI",
            "Name": "Overlaps",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": false
        },
        "OI": {
            "Domain": ["ProperInterval"],
            "Converse": "O",
            "Name": "Overlapped-By",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": false
        },
        "S": {
            "Domain": ["ProperInterval"],
            "Converse": "SI",
            "Name": "Starts",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        },
        "SI": {
            "Domain": ["ProperInterval"],
            "Converse": "S",
            "Name": "Started-By",
            "Range": ["ProperInterval"],
            "Reflexive": false,
            "Symmetric": false,
            "Transitive": true
        }
    },
    "TransTable": {
        "B": {
            "B": "B",
            "BI": "B|BI|D|DI|E|F|FI|M|MI|O|OI|S|SI",
            "D": "B|D|M|O|S",
            "DI": "B",
            "E": "B",
            "F": "B|D|M|O|S",
            "FI": "B",
            "M": "B",
            "MI": "B|D|M|O|S",
            "O": "B",
            "OI": "B|D|M|O|S",
            "S": "B",
            "SI": "B"
        },
        "BI": {
            "B": "B|BI|D|DI|E|F|FI|M|MI|O|OI|S|SI",
            "BI": "BI",
            "D": "BI|D|F|MI|OI",
            "DI": "BI",
            "E": "BI",
            "F": "BI",
            "FI": "BI",
            "M": "BI|D|F|MI|OI",
            "MI": "BI",
            "O": "BI|D|F|MI|OI",
            "OI": "BI",
            "S": "BI|D|F|MI|OI",
            "SI": "BI"
        },
        "D": {
            "B": "B",
            "BI": "BI",
            "D": "D",
            "DI": "B|BI|D|DI|E|F|FI|M|MI|O|OI|S|SI",
            "E": "D",
            "F": "D",
            "FI": "B|D|M|O|S",
            "M": "B",
            "MI": "BI",
            "O": "B|D|M|O|S",
            "OI": "BI|D|F|MI|OI",
            "S": "D",
            "SI": "BI|D|F|MI|OI"
        },
        "DI": {
            "B": "B|DI|FI|M|O",
            "BI": "BI|DI|MI|OI|SI",
            "D": "D|DI|E|F|FI|O|OI|S|SI",
            "DI": "DI",
            "E": "DI",
            "F": "DI|OI|SI",
            "FI": "DI",
            "M": "DI|FI|O",
            "MI": "DI|OI|SI",
            "O": "DI|FI|O",
            "OI": "DI|OI|SI",
            "S": "DI|FI|O",
            "SI": "DI"
        },
        "E": {
            "B": "B",
            "BI": "BI",
            "D": "D",
            "DI": "DI",
            "E": "E",
            "F": "F",
            "FI": "FI",
            "M": "M",
            "MI": "MI",
            "O": "O",
            "OI": "OI",
            "S": "S",
            "SI": "SI"
        },
        "F": {
            "B": "B",
            "BI": "BI",
            "D": "D",
            "DI": "BI|DI|MI|OI|SI",
            "E": "F",
            "F": "F",
            "FI": "E|F|FI",
            "M": "M",
            "MI": "BI",
            "O": "D|O|S",
            "OI": "BI|MI|OI",
            "S": "D",
            "SI": "BI|MI|OI"
        },
        "FI": {
            "B": "B",
            "BI": "BI|DI|MI|OI|SI",
            "D": "D|O|S",
            "DI": "DI",
            "E": "FI",
            "F": "E|F|FI",
            "FI": "FI",
            "M": "M",
            "MI": "DI|OI|SI",
            "O": "O",
            "OI": "DI|OI|SI",
            "S": "O",
            "SI": "DI"
        },
        "M": {
            "B": "B",
            "BI": "BI|DI|MI|OI|SI",
            "D": "D|O|S",
            "DI": "B",
            "E": "M",
            "F": "D|O|S",
            "FI": "B",
            "M": "B",
            "MI": "E|F|FI",
            "O": "B",
            "OI": "D|O|S",
            "S": "M",
            "SI": "M"
        },
        "MI": {
            "B": "B|DI|FI|M|O",
            "BI": "BI",
            "D": "D|F|OI",
            "DI": "BI",
            "E": "MI",
            "F": "MI",
            "FI": "MI",
            "M": "E|S|SI",
            "MI": "BI",
            "O": "D|F|OI",
            "OI": "BI",
            "S": "D|F|OI",
            "SI": "BI"
        },
        "O": {
            "B": "B",
            "BI": "BI|DI|MI|OI|SI",
            "D": "D|O|S",
            "DI": "B|DI|FI|M|O",
            "E": "O",
            "F": "D|O|S",
            "FI": "B|M|O",
            "M": "B",
            "MI": "DI|OI|SI",
            "O": "B|M|O",
            "OI": "D|DI|E|F|FI|O|OI|S|SI",
            "S": "O",
            "SI": "DI|FI|O"
        },
        "OI": {
            "B": "B|DI|FI|M|O",
            "BI": "BI",
            "D": "D|F|OI",
            "DI": "BI|DI|MI|OI|SI",
            "E": "OI",
            "F": "OI",
            "FI": "DI|OI|SI",
            "M": "DI|FI|O",
            "MI": "BI",
            "O": "D|DI|E|F|FI|O|OI|S|SI",
            "OI": "BI|MI|OI",
            "S": "D|F|OI",
            "SI": "BI|MI|OI"
        },
        "S": {
            "B": "B",
            "BI": "BI",
            "D": "D",
            "DI": "B|DI|FI|M|O",
            "E": "S",
            "F": "D",
            "FI": "B|M|O",
            "M": "B",
            "MI": "MI",
            "O": "B|M|O",
            "OI": "D|F|OI",
            "S": "S",
            "SI": "E|S|SI"
        },
        "SI": {
            "B": "B|DI|FI|M|O",
            "BI": "BI",
            "D": "D|F|OI",
            "DI": "DI",
            "E": "SI",
            "F": "OI",
            "FI": "DI",
            "M": "DI|FI|O",
            "MI": "MI",
            "O": "DI|FI|O",
            "OI": "OI",
            "S": "E|S|SI",
            "SI": "SI"
        }
    }
}
'''),
    "RCC8_Algebra.json": json.loads(r'''{
    "Description": "Region Connection Calculus 8 Algebra", 
    "Name": "RCC8_Algebra",
    "Relations": {
        "DC": {
            "Domain": ["Region"],
            "Converse": "DC", 
            "Name": "Disconnected", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": true, 
            "Transitive": false
        }, 
        "EC": {
            "Domain": ["Region"],
            "Converse": "EC", 
            "Name": "ExternallyConnected", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": true, 
            "Transitive": false
        }, 
        "EQ": {
            "Domain": ["Region"],
            "Converse": "EQ", 
            "Name": "Equal", 
            "Range": ["Region"],
            "Reflexive": true, 
            "Symmetric": true, 
            "Transitive": true
        }, 
        "NTPP": {
            "Domain": ["Region"],
            "Converse": "NTPPI", 
            "Name": "NonTangentialProperPart", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": true
        }, 
        "NTPPI": {
            "Domain": ["Region"],
            "Converse": "NTPP", 
            "Name": "NonTangentialProperPartInverse", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": true
        }, 
        "PO": {
            "Domain": ["Region"],
            "Converse": "PO", 
            "Name": "PartiallyOverlapping", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": true, 
            "Transitive": false
        }, 
        "TPP": {
            "Domain": ["Region"],
            "Converse": "TPPI", 
            "Name": "TangentialProperPart", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": false
        }, 
        "TPPI": {
            "Domain": ["Region"],
            "Converse": "TPP", 
            "Name": "TangentialProperPartInverse", 
            "Range": ["Region"],
            "Reflexive": false, 
            "Symmetric": false, 
            "Transitive": false
        }
    }, 
    "TransTable": {
        "DC": {
            "DC": "DC|EC|EQ|NTPP|NTPPI|PO|TPP|TPPI",
            "EC": "DC|EC|NTPP|PO|TPP",
            "EQ": "DC",
            "NTPP": "DC|EC|NTPP|PO|TPP",
            "NTPPI": "DC",
            "PO": "DC|EC|NTPP|PO|TPP",
            "TPP": "DC|EC|NTPP|PO|TPP",
            "TPPI": "DC"
        },
        "EC": {
            "DC": "DC|EC|NTPPI|PO|TPPI",
            "EC": "DC|EC|EQ|PO|TPP|TPPI",
            "EQ": "EC",
            "NTPP": "NTPP|PO|TPP",
            "NTPPI": "DC",
            "PO": "DC|EC|NTPP|PO|TPP",
            "TPP": "EC|NTPP|PO|TPP",
            "TPPI": "DC|EC"
        },
        "EQ": {
            "DC": "DC",
            "EC": "EC",
            "EQ": "EQ",
            "NTPP": "NTPP",
            "NTPPI": "NTPPI",
            "PO": "PO",
            "TPP": "TPP",
            "TPPI": "TPPI"
        },
        "NTPP": {
            "DC": "DC",
            "EC": "DC",
            "EQ": "NTPP",
            "NTPP": "NTPP",
            "NTPPI": "DC|EC|EQ|NTPP|NTPPI|PO|TPP|TPPI",
            "PO": "DC|EC|NTPP|PO|TPP",
            "TPP": "NTPP",
            "TPPI": "DC|EC|NTPP|PO|TPP"
        },
        "NTPPI": {
            "DC": "DC|EC|NTPPI|PO|TPPI",
            "EC": "NTPPI|PO|TPPI",
            "EQ": "NTPPI",
            "NTPP": "EQ|NTPP|NTPPI|PO|TPP|TPPI",
            "NTPPI": "NTPPI",
            "PO": "NTPPI|PO|TPPI",
            "TPP": "NTPPI|PO|TPPI",
            "TPPI": "NTPPI"
        },
        "PO": {
            "DC": "DC|EC|NTPPI|PO|TPPI",
            "EC": "DC|EC|NTPPI|PO|TPPI",
            "EQ": "PO",
            "NTPP": "NTPP|PO|TPP",
            "NTPPI": "DC|EC|NTPPI|PO|TPPI",
            "PO": "DC|EC|EQ|NTPP|NTPPI|PO|TPP|TPPI",
            "TPP": "NTPP|PO|TPP",
            "TPPI": "DC|EC|NTPPI|PO|TPPI"
        },
        "TPP": {
            "DC": "DC",
            "EC": "DC|EC",
            "EQ": "TPP",
            "NTPP": "NTPP",
            "NTPPI": "DC|EC|NTPPI|PO|TPPI",
            "PO": "DC|EC|NTPP|PO|TPP",
            "TPP": "NTPP|TPP",
            "TPPI": "DC|EC|EQ|PO|TPP|TPPI"
        },
        "TPPI": {
            "DC": "DC|EC|NTPPI|PO|TPPI",
            "EC": "EC|NTPPI|PO|TPPI",
            "EQ": "TPPI",
            "NTPP": "NTPP|PO|TPP",
            "NTPPI": "NTPPI",
            "PO": "NTPPI|PO|TPPI",
            "TPP": "EQ|PO|TPP|TPPI",
            "TPPI": "NTPPI|TPPI"
        }
    }
}'''),
}
print("embedded tables:", list(ALGEBRA_TABLE_JSON))

### Algebra class + loaders

`qcn/algebras.py` verbatim — the `Algebra` class (single-relation `compose`/`conv`,
disjunctive-label `compose_sets`/`converse_set`/`intersect`, and the path-composition
*certificate engine in miniature* `compose_path`), the table parser, and the
convex-point-label guard. The **only** change vs. the original: `_load` reads the
embedded `ALGEBRA_TABLE_JSON` instead of a file path (see the `d = ...` line).

In [ ]:
"""Qualitative relation algebras: convex Point, Allen Interval, RCC-8.

Composition (TransTable) and converse maps are loaded from the AUTHORITATIVE
qualreas algebra definitions (alreich/qualreas), bundled in ``algebra_tables/``.
These are the standard tables (Allen-1983 interval composition; Renz & Nebel
RCC-8 composition; linear point algebra).  We map qualreas relation codes to the
canonical codes used throughout this dataset:

  point : '<' '=' '>'
  allen : b bi m mi o oi d di s si f fi eq           (qualreas E -> eq)
  rcc8  : DC EC PO EQ TPP NTPP TPPi NTPPi             (qualreas *I -> *i)

The tables are cross-checked by three INDEPENDENT derivations in
``tests/test_qcn.py``:
  * point composition vs. direct sign reasoning,
  * Allen composition vs. exhaustive endpoint-CSP enumeration,
  * RCC-8 composition soundness vs. exhaustive integer disc-triple enumeration,
plus relation-algebra axioms (identity, involution / Peirce law).
"""
from __future__ import annotations

import json
from itertools import product
from pathlib import Path
from typing import Dict, FrozenSet, List, Tuple

# _TABLE_DIR removed for the notebook: tables are embedded in ALGEBRA_TABLE_JSON above.

# qualreas-code -> canonical-code maps -------------------------------------------------
_POINT_MAP = {"<": "<", "=": "=", ">": ">"}
_ALLEN_MAP = {
    "B": "b", "BI": "bi", "M": "m", "MI": "mi", "O": "o", "OI": "oi",
    "D": "d", "DI": "di", "S": "s", "SI": "si", "F": "f", "FI": "fi", "E": "eq",
}
_RCC8_MAP = {
    "DC": "DC", "EC": "EC", "PO": "PO", "EQ": "EQ",
    "TPP": "TPP", "NTPP": "NTPP", "TPPI": "TPPi", "NTPPI": "NTPPi",
}

# canonical base-relation orderings (stable, used for serialization / histograms)
POINT_BASE = ["<", "=", ">"]
ALLEN_BASE = ["b", "bi", "m", "mi", "o", "oi", "d", "di", "s", "si", "f", "fi", "eq"]
RCC8_BASE = ["DC", "EC", "PO", "EQ", "TPP", "NTPP", "TPPi", "NTPPi"]


class Algebra:
    """A qualitative relation algebra with composition, converse and helpers."""

    def __init__(self, name: str, base: List[str], converse: Dict[str, str],
                 comp: Dict[Tuple[str, str], FrozenSet[str]]):
        self.name = name
        self.base = list(base)
        self.base_set = frozenset(base)
        self.universe = frozenset(base)
        self.converse = dict(converse)
        self.comp = dict(comp)  # (r1, r2) -> frozenset of base relations
        # identity element of the algebra (the relation r with comp(r, x) == {x})
        self.identity = {"point": "=", "allen": "eq", "rcc8": "EQ"}[name]
        # stable index for ordering disjunctive labels
        self._idx = {r: i for i, r in enumerate(base)}

    # -- single-relation operations --------------------------------------------------
    def compose(self, r1: str, r2: str) -> FrozenSet[str]:
        return self.comp[(r1, r2)]

    def conv(self, r: str) -> str:
        return self.converse[r]

    # -- set (disjunctive label) operations ------------------------------------------
    def compose_sets(self, s1, s2) -> FrozenSet[str]:
        """Composition of two disjunctive labels: union of pairwise base comps."""
        out: set = set()
        for r1 in s1:
            for r2 in s2:
                out |= self.comp[(r1, r2)]
        return frozenset(out)

    def converse_set(self, s) -> FrozenSet[str]:
        return frozenset(self.converse[r] for r in s)

    @staticmethod
    def intersect(s1, s2) -> FrozenSet[str]:
        return frozenset(s1) & frozenset(s2)

    def sorted_label(self, s) -> List[str]:
        """Disjunctive label as a list in canonical base order."""
        return [r for r in self.base if r in s]

    # -- path composition (the certificate engine in miniature) ----------------------
    def compose_path(self, rels: List[str]) -> FrozenSet[str]:
        """Iterated composition of a sequence of *atomic* directed relations.

        rels = [rel(n0,n1), rel(n1,n2), ..., rel(n_{k-1},n_k)].
        Returns the composed disjunctive relation between n0 and n_k.
        """
        if not rels:
            return frozenset([self.identity])
        acc: FrozenSet[str] = frozenset([rels[0]])
        for r in rels[1:]:
            acc = self.compose_sets(acc, frozenset([r]))
        return acc


def _parse_transtable(tt: dict, code_map: Dict[str, str]) -> Dict[Tuple[str, str], FrozenSet[str]]:
    comp: Dict[Tuple[str, str], FrozenSet[str]] = {}
    for q1, row in tt.items():
        for q2, val in row.items():
            rels = frozenset(code_map[p] for p in val.split("|"))
            comp[(code_map[q1], code_map[q2])] = rels
    return comp


def _load(name: str, fname: str, code_map: Dict[str, str], base: List[str]) -> Algebra:
    d = ALGEBRA_TABLE_JSON[fname]  # embedded above (original read algebra_tables/<fname>)
    rels = d["Relations"]
    converse = {code_map[q]: code_map[rels[q]["Converse"]] for q in rels}
    comp = _parse_transtable(d["TransTable"], code_map)
    # sanity: full base coverage and square composition table
    assert set(converse) == set(base), (name, "converse keys")
    assert len(comp) == len(base) ** 2, (name, "comp size", len(comp))
    return Algebra(name, base, converse, comp)


def load_algebras() -> Dict[str, Algebra]:
    return {
        "point": _load("point", "Linear_Point_Algebra.json", _POINT_MAP, POINT_BASE),
        "allen": _load("allen", "Linear_Interval_Algebra.json", _ALLEN_MAP, ALLEN_BASE),
        "rcc8": _load("rcc8", "RCC8_Algebra.json", _RCC8_MAP, RCC8_BASE),
    }


# ---------------------------------------------------------------------------------------
# Convex-point-algebra guard: which disjunctive point labels are CONVEX (pointisable).
# Path-consistency is complete only on the convex fragment; the non-convex '!=' ({<,>})
# must never appear in a point-arm disjunctive label.
_CONVEX_POINT_LABELS = [
    frozenset({"<"}), frozenset({"="}), frozenset({">"}),
    frozenset({"<", "="}), frozenset({"=", ">"}), frozenset({"<", "=", ">"}),
]


def is_convex_point_label(s) -> bool:
    return frozenset(s) in _CONVEX_POINT_LABELS


def convex_supersets_of(gold: str) -> List[FrozenSet[str]]:
    """All convex point labels that contain `gold` (sound supersets, no '!=')."""
    return [lab for lab in _CONVEX_POINT_LABELS if gold in lab]


In [ ]:
# module-level algebra registry (as in method.py)
ALG = load_algebras()
for name, alg in ALG.items():
    print(f"{name:6s}  base={alg.base}  identity={alg.identity!r}")

## Model-based realization (`qcn/realize.py`)

Each node is embedded in a concrete geometric model and the gold **atomic** relation is
read directly off the model — so a realizable model is globally consistent *by
construction*, with no solver. `realize()` draws coordinates/endpoints/radii from small
integer pools so that coincidences occur and the degenerate relations (point `=`; Allen
`m/mi/s/si/f/fi/eq`; RCC-8 `EC/TPP/TPPi/EQ`) show up with non-trivial frequency.
`planted_pair()` provides concrete fragments realizing each base relation (used later to
guarantee full relation coverage). Copied verbatim.

In [ ]:
"""Model-based realization of qualitative scenarios.

Each network is built by embedding every node in a concrete geometric model and
reading the gold ATOMIC relation off the model.  A realizable model is, by
construction, GLOBALLY CONSISTENT -- no constraint solver is needed to certify
consistency, and the gold relation on every pair is exact.

  point : node -> integer coordinate x          ; rel = sign(x_i - x_j)
  allen : node -> integer interval (s, e), s<e   ; rel = 13-case endpoint compare
  rcc8  : node -> integer disc (cx, cy, r), r>=1 ; rel via exact d^2 integer compare
          (centers are placed COLLINEAR so that internal/external tangency EC, TPP
           and equality EQ are realized exactly with integer arithmetic; every
           pairwise relation is a genuine RCC-8 relation, so the scenario is a
           valid -- and globally consistent -- RCC-8 model.)

Endpoints / coordinates / radii are drawn from SMALL integer pools so that
coincidences occur and the degenerate relations (point '='; Allen m/mi/s/si/f/fi/eq;
RCC-8 EC/TPP/TPPi/EQ) are realized with non-trivial frequency.
"""
from __future__ import annotations

from typing import List, Sequence, Tuple

import numpy as np

# ----------------------------------------------------------------------------- readers
def point_rel(xi: int, xj: int) -> str:
    if xi < xj:
        return "<"
    if xi > xj:
        return ">"
    return "="


def allen_rel(x: Tuple[int, int], y: Tuple[int, int]) -> str:
    """Allen interval relation of x w.r.t. y. Requires xs<xe and ys<ye."""
    xs, xe = x
    ys, ye = y
    if xe < ys:
        return "b"
    if xe == ys:
        return "m"
    if xs > ye:
        return "bi"
    if xs == ye:
        return "mi"
    # proper overlap guaranteed: xe>ys and xs<ye
    if xs == ys:
        if xe == ye:
            return "eq"
        return "s" if xe < ye else "si"
    if xe == ye:
        return "fi" if xs < ys else "f"
    if xs < ys:
        return "o" if xe < ye else "di"
    # xs > ys
    return "d" if xe < ye else "oi"


def rcc8_rel(a: Tuple[int, int, int], b: Tuple[int, int, int]) -> str:
    """RCC-8 relation of disc a w.r.t. disc b using exact integer arithmetic."""
    ax, ay, ar = a
    bx, by, br = b
    d2 = (ax - bx) ** 2 + (ay - by) ** 2
    s2 = (ar + br) ** 2
    df = ar - br
    df2 = df * df
    if d2 > s2:
        return "DC"
    if d2 == s2:
        return "EC"
    # d2 < s2 : overlap or containment
    if d2 == 0 and ar == br:
        return "EQ"
    if d2 < df2:
        return "NTPP" if ar < br else "NTPPi"
    if d2 == df2:  # internal tangency (ar != br here, else EQ already returned)
        return "TPP" if ar < br else "TPPi"
    return "PO"


_READERS = {"point": point_rel, "allen": allen_rel, "rcc8": rcc8_rel}


def relation(algebra: str, oi, oj) -> str:
    """Gold atomic relation between two embedded objects (directed: oi rel oj)."""
    return _READERS[algebra](oi, oj)


# ------------------------------------------------------------------------- realization
def realize(algebra: str, n_nodes: int, rng: np.random.RandomState) -> List:
    """Return a list of embedded objects (one per node) drawn from small integer pools.

    Pools are sized so endpoint/coordinate coincidences -- hence degenerate relations
    -- occur with non-trivial probability.
    """
    if algebra == "point":
        pool = max(3, int(round(1.6 * n_nodes)))  # collisions -> '=' appears
        return [int(v) for v in rng.randint(0, pool + 1, size=n_nodes)]

    if algebra == "allen":
        # draw two distinct integer time points from a small pool; coincident endpoints
        # across nodes realize m/mi/s/si/f/fi/eq.
        pts = max(4, int(round(1.4 * n_nodes)))
        out: List[Tuple[int, int]] = []
        for _ in range(n_nodes):
            a = int(rng.randint(0, pts))
            b = int(rng.randint(0, pts))
            while a == b:
                b = int(rng.randint(0, pts))
            out.append((min(a, b), max(a, b)))
        return out

    if algebra == "rcc8":
        # collinear discs: center on x-axis (cy=0), radius>=1, both from small pools.
        cpool = max(4, int(round(1.5 * n_nodes)))
        rpool = max(2, n_nodes // 2 + 1)
        out2: List[Tuple[int, int, int]] = []
        for _ in range(n_nodes):
            cx = int(rng.randint(0, cpool + 1))
            r = int(rng.randint(1, rpool + 1))
            out2.append((cx, 0, r))
        return out2

    raise ValueError(f"unknown algebra {algebra!r}")


# -------------------------------------------------------------- explicit relation planting
# Concrete model fragments that realize EACH base relation, used to guarantee global
# coverage of all base relations even if random sampling under-represents a degenerate one.
def planted_pair(algebra: str, rel: str):
    """Return a concrete (obj_i, obj_j) pair whose gold relation == rel (verified)."""
    if algebra == "point":
        table = {"<": (0, 1), "=": (1, 1), ">": (1, 0)}
        return table[rel]
    if algebra == "allen":
        table = {
            "b": ((0, 1), (2, 3)), "bi": ((2, 3), (0, 1)),
            "m": ((0, 1), (1, 2)), "mi": ((1, 2), (0, 1)),
            "o": ((0, 2), (1, 3)), "oi": ((1, 3), (0, 2)),
            "d": ((1, 2), (0, 3)), "di": ((0, 3), (1, 2)),
            "s": ((0, 1), (0, 2)), "si": ((0, 2), (0, 1)),
            "f": ((1, 2), (0, 2)), "fi": ((0, 2), (1, 2)),
            "eq": ((0, 1), (0, 1)),
        }
        return table[rel]
    if algebra == "rcc8":
        # collinear discs (cx, 0, r)
        table = {
            "DC": ((0, 0, 1), (5, 0, 1)),
            "EC": ((0, 0, 2), (5, 0, 3)),       # d=5 == 2+3
            "PO": ((0, 0, 3), (4, 0, 3)),       # |0|<4<6
            "EQ": ((0, 0, 2), (0, 0, 2)),
            "TPP": ((1, 0, 1), (0, 0, 2)),      # d=1 == 2-1, a inside b tangent
            "NTPP": ((0, 0, 1), (0, 0, 3)),     # d=0 < 3-1
            "TPPi": ((0, 0, 2), (1, 0, 1)),
            "NTPPi": ((0, 0, 3), (0, 0, 1)),
        }
        return table[rel]
    raise ValueError(algebra)


## Constraint-graph topology (`qcn/topology.py`)

The controlled design. Every generator returns `(G, s, t)` with the held-out query edge
`(s,t)` **excluded**. Three families:

- **theta**: `P` internally vertex-disjoint `s→t` paths of length `L` ⇒ redundancy `P`,
  hop `L`, cyclomatic `mu = P−1`.
- **cyclo_aug**: theta base + chords between intermediates of *different* paths (each
  chord raises `mu` and creates new `s→t` paths — the cross-links that let *iterated*
  propagation tighten the query beyond one naive pass).
- **random_andl**: Renz & Nebel `A(n,d)` random network; query = a far-apart non-adjacent
  pair.

Copied verbatim. (A small `topology` namespace is created at the end so the downstream
`method.py` code keeps its original `topology.theta(...)` call style.)

In [ ]:
"""Constraint-graph topology generators (the controlled design).

Every generator returns ``(G, s, t)`` where ``G`` is an undirected ``networkx``
graph on integer nodes ``0..V-1`` that EXCLUDES the held-out query edge ``(s, t)``
(the query relation starts universal and is obtainable only by composing >=1
multi-edge path).  The geometric embedding + gold relations are added later by
``method.py``; topology here is purely structural.

Cyclomatic number used downstream:  mu = E - V + C  (C = #connected components).

FAMILY 1  theta        : P internally vertex-disjoint s->t paths of length L
                         => redundancy=P, hop=L, V=P*(L-1)+2, derived mu=P-1.
FAMILY 2  cyclo_aug    : theta base + chords between intermediates of DIFFERENT
                         paths (each chord on a connected graph raises mu by 1 and
                         generally creates new s->t paths -- the cross-links that let
                         ITERATED propagation tighten the query beyond one pass).
FAMILY 3  random_andl  : Renz & Nebel A(n,d,l) random network, avg degree d; query =
                         a non-adjacent connected pair (deduction-required).
"""
from __future__ import annotations

from typing import List, Tuple

import networkx as nx
import numpy as np


def theta(rng: np.random.RandomState, P: int, L: int) -> Tuple[nx.Graph, int, int]:
    """P internally vertex-disjoint s->t paths, each with L edges (L-1 intermediates)."""
    assert L >= 2, "theta needs L>=2 so s,t share no direct edge"
    G = nx.Graph()
    s, t = 0, 1
    G.add_nodes_from([s, t])
    nxt = 2
    for _ in range(P):
        prev = s
        for j in range(L - 1):
            node = nxt
            nxt += 1
            G.add_node(node)
            G.add_edge(prev, node)
            prev = node
        G.add_edge(prev, t)
    return G, s, t


def cyclo_aug(rng: np.random.RandomState, P: int, L: int, n_chords: int
              ) -> Tuple[nx.Graph, int, int]:
    """theta(P,L) base + `n_chords` chords between intermediates of different paths."""
    G, s, t = theta(rng, P, L)
    # group intermediates by path
    paths_mids: List[List[int]] = []
    node = 2
    for _ in range(P):
        paths_mids.append(list(range(node, node + (L - 1))))
        node += (L - 1)
    # candidate chords: intermediate pairs on DIFFERENT paths, not already edges,
    # never touching s or t (so we don't shortcut the query).
    cands = []
    for pi in range(P):
        for pj in range(pi + 1, P):
            for u in paths_mids[pi]:
                for v in paths_mids[pj]:
                    if not G.has_edge(u, v):
                        cands.append((u, v))
    rng.shuffle(cands)
    for (u, v) in cands[:n_chords]:
        G.add_edge(u, v)
    return G, s, t


def single_path(rng: np.random.RandomState, L: int) -> Tuple[nx.Graph, int, int]:
    """A single s->t chain of length L (redundancy 1, mu 0). Used for the mu=0 cell."""
    return theta(rng, 1, L)


def add_distractors(G: nx.Graph, s: int, t: int, rng: np.random.RandomState, k: int
                    ) -> nx.Graph:
    """Attach k distractor nodes as a random tree bridged to a non-terminal core node.

    Adds nodes/edges (inflating node count) WITHOUT creating any new s->t path:
    distractors form a tree, bridged to the core by a single edge at an intermediate
    node (never s or t), so no alternative s->t route is introduced.
    """
    if k <= 0:
        return G
    core_internal = [n for n in G.nodes if n not in (s, t)]
    anchor = int(rng.choice(core_internal)) if core_internal else s
    start = max(G.nodes) + 1
    new_nodes = list(range(start, start + k))
    G.add_node(new_nodes[0])
    G.add_edge(anchor, new_nodes[0])  # single bridge (tree edge, no new s-t path)
    for i in range(1, k):
        parent = int(rng.choice(new_nodes[:i]))
        G.add_node(new_nodes[i])
        G.add_edge(parent, new_nodes[i])
    return G


def random_andl(rng: np.random.RandomState, n: int, d: int, max_tries: int = 200
                ) -> Tuple[nx.Graph, int, int]:
    """Renz & Nebel A(n,d): G(n, p=d/(n-1)); query = far-apart non-adjacent pair.

    Returns the largest connected component (relabeled 0..m-1) with a designated
    non-adjacent query pair (s,t) at graph distance >=2 (deduction-required).

    Edge probability is capped strictly below 1 ((n-2)/(n-1)) so the graph is never
    complete -- a deduction-required (non-adjacent) query pair always exists.  When the
    requested average degree d >= n-1 this caps realized density below target; the
    realized average degree is recorded per row (num_edges/num_nodes).
    """
    p = min(d / (n - 1), (n - 2) / (n - 1))
    for _ in range(max_tries):
        seed = int(rng.randint(0, 2 ** 31 - 1))
        G0 = nx.gnp_random_graph(n, p, seed=seed)
        if G0.number_of_edges() == 0:
            continue
        comp = max(nx.connected_components(G0), key=len)
        if len(comp) < 4:
            continue
        H = nx.convert_node_labels_to_integers(G0.subgraph(comp).copy())
        # all non-adjacent pairs reachable at distance >=2
        nonadj = [(u, v) for u in H.nodes for v in H.nodes
                  if u < v and not H.has_edge(u, v)]
        if not nonadj:
            continue
        # choose the pair with the largest shortest-path distance (deeper deduction)
        best = None
        best_d = -1
        # sample up to 60 candidate pairs for speed on dense graphs
        idx = rng.permutation(len(nonadj))[:60]
        for i in idx:
            u, v = nonadj[i]
            try:
                dd = nx.shortest_path_length(H, u, v)
            except nx.NetworkXNoPath:
                continue
            if dd > best_d:
                best_d, best = dd, (u, v)
        if best is None:
            continue
        s, t = best
        # relabel so s=0, t=1 for consistency with theta (optional but tidy)
        return H, int(s), int(t)
    raise RuntimeError(f"random_andl: no valid network for n={n} d={d}")



# expose the topology functions under a `topology` namespace so the original method.py
# code below can keep calling `topology.theta(...)`, `topology.random_andl(...)`, etc.
topology = types.SimpleNamespace(
    theta=theta, cyclo_aug=cyclo_aug, single_path=single_path,
    add_distractors=add_distractors, random_andl=random_andl)

## Natural-language realization (`qcn/verbalize.py`)

Each **non-query** edge becomes one professional-prose sentence describing its gold
relation (2–3 paraphrases per relation, recorded per edge for error analysis). The
held-out query edge is never verbalized; the document ends with a single `Query:` line.
Entities are unique within a network, drawn from pools of ≥50 temporal/spatial phrases.
Copied verbatim. (A `verbalize` namespace is created at the end, as for topology.)

In [ ]:
"""Template NL realization of a qualitative constraint network.

Each NON-query edge is verbalized as ONE professional-prose sentence describing its
gold relation (chosen among 2-3 paraphrases, recorded per edge for error analysis).
The held-out query edge is NEVER verbalized; the document ends with a single
``Query:`` line.  Entities are unique within a network and drawn from pools of >=50
distinct phrases (temporal for point/allen, spatial for rcc8).
"""
from __future__ import annotations

from typing import Dict, List, Tuple

import numpy as np

# ---------------------------------------------------------------- entity phrase pools
TEMPORAL_POOL: List[str] = [
    "the merger announcement", "the board meeting", "the product launch",
    "the regulatory filing", "the press briefing", "the quarterly audit",
    "the shareholder vote", "the funding round", "the site inspection",
    "the contract signing", "the staff onboarding", "the policy rollout",
    "the data migration", "the system outage", "the security patch",
    "the marketing campaign", "the earnings call", "the budget review",
    "the hiring freeze", "the office relocation", "the pilot program",
    "the customer survey", "the compliance training", "the vendor negotiation",
    "the prototype demo", "the field trial", "the litigation hearing",
    "the patent submission", "the supplier handover", "the inventory count",
    "the network upgrade", "the recruitment drive", "the strategy offsite",
    "the certification review", "the maintenance window", "the price adjustment",
    "the partnership signing", "the recall notice", "the warehouse expansion",
    "the software release", "the tax assessment", "the safety drill",
    "the procurement cycle", "the design review", "the investor roadshow",
    "the membership renewal", "the catalog refresh", "the loan disbursement",
    "the depot closure", "the franchise opening", "the trademark dispute",
    "the seasonal promotion", "the freight shipment", "the annual report",
    "the holiday shutdown", "the equipment delivery", "the licensing deal",
    "the curriculum update", "the grant disbursal", "the venue booking",
]

SPATIAL_POOL: List[str] = [
    "the industrial zone", "the flood plain", "the conservation area",
    "the harbor district", "Parcel A", "Parcel B", "the buffer strip",
    "the wetland reserve", "the transit corridor", "the commercial block",
    "the residential tract", "the quarry site", "the green belt",
    "the watershed basin", "the customs enclave", "the parking precinct",
    "the warehouse lot", "the heritage district", "the airport perimeter",
    "the coastal margin", "the forest compartment", "the mining lease",
    "the irrigation district", "the campus grounds", "the utility easement",
    "the protected habitat", "the floodway channel", "the survey block",
    "the municipal park", "the freight terminal", "the orchard plot",
    "the recreation ground", "the levee district", "the railway reserve",
    "the cemetery grounds", "the pasture allotment", "the marina basin",
    "the vineyard estate", "the construction site", "the landfill cell",
    "the nature sanctuary", "the harbor jetty", "the school catchment",
    "the power substation", "the grazing common", "the riverside walk",
    "the market square", "the depot yard", "the sports complex",
    "the embassy compound", "the shopping precinct", "the fishing ground",
    "the tunnel approach", "the meadow enclosure", "the dockland estate",
    "the highway verge", "the plantation block", "the reservoir margin",
    "the village green", "the customs yard",
]

# ---------------------------------------------------------------- templates per relation
POINT_TEMPLATES: Dict[str, List[str]] = {
    "<": ["{A} occurred before {B}.", "{A} took place earlier than {B}.",
          "{A} came before {B}."],
    "=": ["{A} happened at the same time as {B}.",
          "{A} occurred simultaneously with {B}.", "{A} and {B} coincided in time."],
    ">": ["{A} occurred after {B}.", "{A} took place later than {B}.",
          "{A} came after {B}."],
}

ALLEN_TEMPLATES: Dict[str, List[str]] = {
    "b": ["{A} ended before {B} began.", "{A} was entirely over before {B} started."],
    "bi": ["{A} began only after {B} had ended.",
           "{A} started after {B} was completely finished."],
    "m": ["{A} ended exactly as {B} began.",
          "{A} finished at the very moment {B} started."],
    "mi": ["{A} began exactly as {B} ended.",
           "{A} started at the very moment {B} finished."],
    "o": ["{A} began before {B} and the two overlapped, with {A} finishing first.",
          "{A} started first and was still ongoing when {B} began, yet {A} ended before {B} did."],
    "oi": ["{A} began after {B} and the two overlapped, with {A} finishing later.",
           "{B} started first and was still ongoing when {A} began, and {A} ended after {B} did."],
    "d": ["{A} took place entirely during {B}.",
          "{A} happened wholly within the span of {B}."],
    "di": ["{A} began before {B} and ended after it, spanning {B} entirely.",
           "{B} took place entirely during {A}."],
    "s": ["{A} and {B} began together, but {A} ended first.",
          "{A} and {B} started at the same time, with {A} finishing earlier."],
    "si": ["{A} and {B} began together, but {A} ended later.",
           "{A} and {B} started at the same time, with {A} finishing later."],
    "f": ["{A} and {B} ended together, but {A} began later.",
          "{A} and {B} finished at the same time, with {A} starting later."],
    "fi": ["{A} and {B} ended together, but {A} began earlier.",
           "{A} and {B} finished at the same time, with {A} starting earlier."],
    "eq": ["{A} and {B} spanned exactly the same period.",
           "{A} and {B} occurred over precisely the same interval."],
}

RCC8_TEMPLATES: Dict[str, List[str]] = {
    "DC": ["{A} is completely separate from {B}.",
           "{A} and {B} are disconnected, sharing no points."],
    "EC": ["{A} touches {B} along its boundary without overlapping.",
           "{A} and {B} meet at their boundary but share no interior."],
    "PO": ["{A} and {B} partially overlap.",
           "{A} and {B} share a common region while each also extends beyond the other."],
    "EQ": ["{A} occupies exactly the same area as {B}.",
           "{A} and {B} cover precisely the same region."],
    "TPP": ["{A} lies inside {B} and touches its boundary.",
            "{A} is contained within {B}, meeting {B}'s outer edge."],
    "NTPP": ["{A} lies strictly inside {B}.",
             "{A} is contained well within {B}, away from its boundary."],
    "TPPi": ["{A} contains {B}, which touches {A}'s boundary.",
             "{B} lies inside {A} and meets {A}'s boundary."],
    "NTPPi": ["{A} strictly contains {B}.", "{B} lies strictly inside {A}."],
}

_TEMPLATES = {"point": POINT_TEMPLATES, "allen": ALLEN_TEMPLATES, "rcc8": RCC8_TEMPLATES}
_POOL = {"point": TEMPORAL_POOL, "allen": TEMPORAL_POOL, "rcc8": SPATIAL_POOL}
_QUERY_KIND = {"point": "temporal", "allen": "temporal", "rcc8": "spatial"}


def assign_entities(algebra: str, nodes: List[int], rng: np.random.RandomState
                    ) -> Dict[int, str]:
    pool = _POOL[algebra]
    if len(nodes) > len(pool):
        raise ValueError(f"network has {len(nodes)} nodes > pool {len(pool)}")
    chosen = rng.choice(len(pool), size=len(nodes), replace=False)
    return {int(n): pool[int(i)] for n, i in zip(nodes, chosen)}


def realize_text(algebra: str, edges: List[Tuple[int, int, str]],
                 entity_map: Dict[int, str], s: int, t: int,
                 rng: np.random.RandomState) -> Tuple[str, Dict[str, int]]:
    """Return (document, templates_used). Each edge -> one sentence; query is a final line."""
    tmpl = _TEMPLATES[algebra]
    sentences: List[str] = []
    templates_used: Dict[str, int] = {}
    order = list(range(len(edges)))
    rng.shuffle(order)
    for ei in order:
        u, v, rel = edges[ei]
        assert {u, v} != {s, t}, "query pair must not be a verbalized edge"
        paraphrases = tmpl[rel]
        idx = int(rng.randint(0, len(paraphrases)))
        templates_used[f"{u}-{v}"] = idx
        # format with raw phrases, then capitalize ONLY the sentence-initial character
        # (so mid-sentence entity repeats stay lowercase, e.g. "..., but the offsite ...").
        sentence = _cap(paraphrases[idx].format(A=entity_map[u], B=entity_map[v]))
        sentences.append(sentence)
    query_line = (f"Query: what is the {_QUERY_KIND[algebra]} relation between "
                  f"{entity_map[s]} and {entity_map[t]}?")
    document = " ".join(sentences) + "\n" + query_line
    return document, templates_used


def _cap(phrase: str) -> str:
    """Capitalize the first character for sentence-initial position (keeps proper names)."""
    return phrase[0].upper() + phrase[1:] if phrase else phrase



# expose under a `verbalize` namespace for the original method.py call style.
verbalize = types.SimpleNamespace(
    assign_entities=assign_entities, realize_text=realize_text)

## The generator (`method.py`)

### The structural grid

27 structural cells per algebra, sweeping redundancy `P`, hop `L`, cyclomatic `mu`,
node-count regime, and random `A(n,d)`. `build_grid()` and `cell_labels()` copied
verbatim.

In [ ]:
def build_grid():
    """Return {algebra: [cell, ...]}. Same 27 structural cells per algebra."""
    cells = []
    # (i) REDUNDANCY sweep (Family 1 theta)
    for P in (1, 2, 3, 4, 6, 8):
        cells.append(dict(cell_id=f"red_P{P}_L2", family="theta", P=P, L=2))
    for P in (1, 2, 3, 4):
        cells.append(dict(cell_id=f"red_P{P}_L3", family="theta", P=P, L=3))
    # (ii) HOP sweep (Family 1 theta) at P=2
    for L in (2, 3, 4, 5):
        cells.append(dict(cell_id=f"hop_L{L}_P2", family="theta", P=2, L=L))
    # (iii) CYCLOMATIC sweep (Family 2) from a P=2,L=3 base
    for mu in (0, 1, 2, 3):
        cells.append(dict(cell_id=f"cyc_mu{mu}", family="cyclo_aug", P=2, L=3,
                          cyclomatic_target=mu))
    # (iv) NODE-COUNT robustness (Family 1 theta P=3,L=3 + distractors)
    for regime, k in (("small", 0), ("medium", 8), ("large", 20)):
        cells.append(dict(cell_id=f"nodes_{regime}", family="theta_distract", P=3, L=3,
                          k=k, node_count_regime=regime))
    # (v) RANDOM Renz-Nebel A(n,d) (Family 3)
    for n in (8, 12):
        for d in (3, 6, 9):
            cells.append(dict(cell_id=f"rand_n{n}_d{d}", family="random_andl", n=n, d=d))
    return {alg: [dict(c) for c in cells] for alg in ("point", "allen", "rcc8")}


def cell_labels(cell):
    """Normalized cell-label block recorded on every row."""
    return {
        "cell_id": cell["cell_id"],
        "generator_family": cell["family"],
        "redundancy_P": cell.get("P"),
        "hop_count_L": cell.get("L"),
        "cyclomatic_target": cell.get("cyclomatic_target"),
        "node_count_regime": cell.get("node_count_regime"),
        "n": cell.get("n"),
        "d": cell.get("d"),
    }

### Per-cell graph construction + helpers

`build_graph()` dispatches a cell to its topology family; `reference_label()` produces a
*sound* disjunctive superset of the gold relation (convenience only — the canonical
ground truth is the gold atomic graph); `fold_of()` assigns the deterministic
pilot/dev/test split via `md5(seed)%100`. Copied verbatim.

In [ ]:
def build_graph(cell, rng):
    fam = cell["family"]
    if fam == "theta":
        return topology.theta(rng, cell["P"], cell["L"])
    if fam == "cyclo_aug":
        if cell["cyclomatic_target"] == 0:
            return topology.single_path(rng, cell["L"])
        return topology.cyclo_aug(rng, cell["P"], cell["L"], cell["cyclomatic_target"] - 1)
    if fam == "theta_distract":
        G, s, t = topology.theta(rng, cell["P"], cell["L"])
        G = topology.add_distractors(G, s, t, rng, cell["k"])
        return G, s, t
    if fam == "random_andl":
        return topology.random_andl(rng, cell["n"], cell["d"])
    raise ValueError(fam)


def reference_label(algebra, gold, rng):
    """A SOUND disjunctive superset of `gold` (avg size REF_LABEL_L). Convex for point."""
    if algebra == "point":
        opts = convex_supersets_of(gold)
        return ALG[algebra].sorted_label(opts[int(rng.randint(0, len(opts)))])
    base = ALG[algebra].base
    l = REF_LABEL_L[algebra]
    size = int(np.clip(round(rng.normal(l, 1.5)), 1, len(base)))
    others = [r for r in base if r != gold]
    rng.shuffle(others)
    chosen = set([gold]) | set(others[: size - 1])
    return ALG[algebra].sorted_label(chosen)


def fold_of(seed):
    h = int(hashlib.md5(str(seed).encode()).hexdigest(), 16) % 100
    return "pilot" if h < 10 else ("dev" if h < 30 else "test")

### Build one network (with the correctness gate)

`build_network()` embeds the graph and `_finish()` does the heavy lifting: read gold
atomic relations off the model, measure structure (`mu`, cycle basis, simple `s→t`
paths), enumerate capped deduction paths, and — the heart of the dataset — assert the
**correctness gate**: for every enumerated `s→t` path, the composition of the gold atomic
relations along it *must contain* the gold query relation. It also computes the one-pass
certificate (`naive_intersection`, `has_bite`, `singleton_resolved`) and emits the full
`exp_sel_data_out` row. Copied verbatim.

In [ ]:
def build_network(algebra, cell, seed, index):
    rng = np.random.RandomState(seed)
    G, s, t = build_graph(cell, rng)
    nodes = sorted(G.nodes())
    objs_list = realize(algebra, len(nodes), rng)
    objs = {n: objs_list[i] for i, n in enumerate(nodes)}
    return _finish(algebra, cell, seed, G, s, t, objs, nodes, rng)


def _finish(algebra, cell, seed, G, s, t, objs, nodes, rng):
    alg = ALG[algebra]
    # gold atomic relations (canonical orientation u<v), query held out
    gold_edges = []
    for (u, v) in G.edges():
        a, b = (u, v) if u < v else (v, u)
        gold_edges.append((a, b, relation(algebra, objs[a], objs[b])))
    gold_edges.sort()
    gold_query = relation(algebra, objs[s], objs[t])

    # structural measures
    V, E = G.number_of_nodes(), G.number_of_edges()
    C = nx.number_connected_components(G)
    mu = E - V + C
    cycle_basis_size = len(nx.cycle_basis(G))
    assert cycle_basis_size == mu, (cell["cell_id"], "mu mismatch", mu, cycle_basis_size)

    # enumerate (capped) deduction paths s->t
    raw_paths = list(islice(nx.all_simple_paths(G, s, t, cutoff=LEN_CUTOFF), PATH_CAP + 1))
    paths_truncated = len(raw_paths) > PATH_CAP
    paths = raw_paths[:PATH_CAP]
    assert len(paths) >= 1, (cell["cell_id"], "deduction-required: no s-t path")

    # per-path composition + correctness GATE
    path_compositions = []
    contributing_edges = set()
    naive = set(alg.base)  # universe
    for path in paths:
        rels = [relation(algebra, objs[path[i]], objs[path[i + 1]])
                for i in range(len(path) - 1)]
        composed = alg.compose_path(rels)
        # GATE: realizable scenario => gold query must lie in every path's composition
        assert gold_query in composed, (
            algebra, cell["cell_id"], seed, "GATE FAIL", path, rels, sorted(composed))
        path_compositions.append(alg.sorted_label(composed))
        naive &= composed
        for i in range(len(path) - 1):
            a, b = path[i], path[i + 1]
            contributing_edges.add((a, b) if a < b else (b, a))
    assert gold_query in naive, "naive intersection lost gold query"
    has_bite = frozenset(naive) != alg.universe
    singleton_resolved = frozenset(naive) == frozenset([gold_query])

    # reference SOUND disjunctive labels (convenience; canonical GT is the atomic graph)
    ref_labels = [[a, b, reference_label(algebra, r, rng)] for (a, b, r) in gold_edges]

    # NL realization
    entity_map = verbalize.assign_entities(algebra, nodes, rng)
    edges_for_text = [(a, b, r) for (a, b, r) in gold_edges]
    document, templates_used = verbalize.realize_text(
        algebra, edges_for_text, entity_map, s, t, rng)
    # deduction-required guard (structural): the query pair is never a verbalized edge,
    # so the two query entities never co-occur in any single sentence.
    assert not G.has_edge(s, t), "deduction-required violated: query pair is an edge"

    # serialized gold graph (output is a STRING per schema)
    out_obj = {
        "edges": [{"source": a, "target": b, "relation": r} for (a, b, r) in gold_edges],
        "query_edge": {"source": s, "target": t, "relation": gold_query, "is_query": True},
    }
    output_str = json.dumps(out_obj, separators=(",", ":"))

    # compact model embedding aligned to abstract-graph node order
    if algebra == "point":
        model = [int(objs[n]) for n in nodes]
    elif algebra == "allen":
        model = [[int(objs[n][0]), int(objs[n][1])] for n in nodes]
    else:
        model = [[int(objs[n][0]), int(objs[n][1]), int(objs[n][2])] for n in nodes]

    row = {
        "input": document,
        "output": output_str,
        "metadata_fold": fold_of(seed),
        "metadata_algebra": algebra,
        "metadata_cell": cell_labels(cell),
        "metadata_query": {"source": s, "target": t, "relation": gold_query},
        "metadata_structure": {
            "num_nodes": V, "num_edges": E, "avg_degree": round(2 * E / V, 3),
            "cyclomatic_number": mu, "cycle_basis_size": cycle_basis_size,
            "num_simple_paths_s_t": len(paths), "paths_truncated": paths_truncated,
            "contributing_edge_count": len(contributing_edges),
            "len_cutoff": LEN_CUTOFF, "path_cap": PATH_CAP,
        },
        "metadata_paths": {
            "path_list": [list(map(int, p)) for p in paths],
            "path_compositions": path_compositions,
            "naive_intersection": alg.sorted_label(naive),
            "has_bite": bool(has_bite),
            "singleton_resolved": bool(singleton_resolved),
        },
        "metadata_abstract_graph": {
            "nodes": [int(n) for n in nodes],
            "edges": [[a, b, r] for (a, b, r) in gold_edges],
            "query_edge": [int(s), int(t)],
        },
        "metadata_reference_disjunctive_labels": ref_labels,
        "metadata_entity_map": {str(n): entity_map[n] for n in nodes},
        "metadata_templates_used": templates_used,
        "metadata_model_embedding": model,
        "metadata_seed": int(seed),
    }
    stats = {
        "algebra": algebra, "cell_id": cell["cell_id"], "fold": row["metadata_fold"],
        "rel_counts": Counter([r for (_, _, r) in gold_edges] + [gold_query]),
        "has_bite": bool(has_bite), "singleton_resolved": bool(singleton_resolved),
        "mu": mu, "num_paths": len(paths), "num_nodes": V,
    }
    return row, stats

### Deterministic seeds, relation-coverage planting, and dataset QA

`net_seed()` gives a reproducible/resumable per-`(algebra, cell, index)` seed;
`build_coverage_row()` plants a minimal network guaranteeing a given base relation
appears (so global relation coverage is guaranteed even if random sampling misses a
degenerate relation); `build_qa()` assembles the dataset-level provenance / coverage /
`cell_summary` metadata. Copied verbatim.

In [ ]:
def net_seed(algebra, cell_id, i):
    """Deterministic 32-bit seed per (algebra, cell, network-index).

    Independent of per-cell counts -> reproducible AND resumable (changing
    --networks-* does not perturb the seeds of already-generated networks).
    """
    return int(hashlib.md5(f"{algebra}|{cell_id}|{i}".encode()).hexdigest()[:8], 16)


def _line_graph_3():
    G = nx.Graph()
    G.add_edges_from([(0, 2), (2, 1)])  # s=0, t=1, intermediate=2
    return G


def build_coverage_row(algebra, rel, seed):
    """Minimal planted network guaranteeing base relation `rel` appears in the corpus.

    s=0 -m=2- t=1 chain; the edge (0,2) is forced (via a concrete model fragment) to
    realize `rel`; node 1 is placed far away so (2,1) is a clean separating relation.
    Routed through the SAME metadata pipeline (_finish) as ordinary networks.
    """
    G, s, t = _line_graph_3(), 0, 1
    oi, oj = planted_pair(algebra, rel)  # relation(oi, oj) == rel  -> assign to (0, 2)
    if algebra == "point":
        objs = {0: int(oi), 2: int(oj), 1: int(max(oi, oj)) + 5}
    elif algebra == "allen":
        shift = max(oi[1], oj[1]) + 10
        objs = {0: tuple(oi), 2: tuple(oj), 1: (shift, shift + 2)}
    else:
        shift = max(oi[0], oj[0]) + 100
        objs = {0: tuple(oi), 2: tuple(oj), 1: (shift, 0, 1)}
    cell = dict(cell_id=f"coverage_{rel}", family="coverage", P=1, L=2)
    return _finish(algebra, cell, seed, G, s, t, objs, sorted(G.nodes()),
                   np.random.RandomState(seed))[0]


def build_qa(all_stats, rel_hist, rows, args):
    """Dataset-level metadata: provenance, coverage tables, histograms, pre-registration."""
    per_cell = defaultdict(lambda: {"count": 0, "has_bite": 0, "singleton": 0,
                                    "mu": Counter(), "paths": [], "nodes": []})
    fold = defaultdict(Counter)
    for st in all_stats:
        key = f"{st['algebra']}/{st['cell_id']}"
        c = per_cell[key]
        c["count"] += 1
        c["has_bite"] += int(st["has_bite"])
        c["singleton"] += int(st["singleton_resolved"])
        c["mu"][st["mu"]] += 1
        c["paths"].append(st["num_paths"])
        c["nodes"].append(st["num_nodes"])
        fold[st["algebra"]][st["fold"]] += 1

    cell_summary = {}
    for key, c in sorted(per_cell.items()):
        n = c["count"]
        cell_summary[key] = {
            "count": n,
            "has_bite_frac": round(c["has_bite"] / n, 4),
            "singleton_resolved_frac": round(c["singleton"] / n, 4),
            "mu_distribution": dict(sorted(c["mu"].items())),
            "mean_num_simple_paths": round(float(np.mean(c["paths"])), 3),
            "mean_num_nodes": round(float(np.mean(c["nodes"])), 3),
        }

    return {
        "name": "Synthetic QCN Backbone",
        "description": (
            "Controlled, globally-consistent qualitative constraint networks over three "
            "relation algebras (convex Point, Allen Interval, RCC-8) built by model-based "
            "realization (integer points / integer-grid intervals / collinear integer "
            "discs), with controlled query topology (redundancy P, hop L, cyclomatic mu, "
            "node count, random Renz-Nebel A(n,d)) and template NL realizations. The "
            "canonical ground truth is the gold ATOMIC graph; every network is realizable "
            "hence globally consistent, and the gold relation on each edge is exact."),
        "algebras": {
            "point": {"role": "primary", "base_relations": ALG["point"].base,
                      "convex_only": True,
                      "note": "non-convex '!=' ({<,>}) forbidden in disjunctive labels"},
            "allen": {"role": "primary", "base_relations": ALG["allen"].base},
            "rcc8": {"role": "secondary", "base_relations": ALG["rcc8"].base,
                     "embedding": "collinear integer discs (exact integer tangency)"},
        },
        "composition_tables": {
            "source": "alreich/qualreas (authoritative TransTable + Converse)",
            "files": ["Linear_Point_Algebra.json", "Linear_Interval_Algebra.json",
                      "RCC8_Algebra.json"],
            "verification": ("Cross-checked in tests/test_qcn.py by 3 independent "
                             "derivations: point composition vs direct sign reasoning; "
                             "Allen composition vs exhaustive endpoint-CSP enumeration "
                             "(full table match); RCC-8 reader soundness vs disc-triple "
                             "enumeration; plus identity + converse-distributivity axioms. "
                             "All 436 checks pass."),
        },
        "generation": {
            "tag": args.tag,
            "networks_per_cell_primary": args.networks_primary,
            "networks_per_cell_secondary": args.networks_secondary,
            "path_cap": PATH_CAP, "len_cutoff": LEN_CUTOFF,
            "total_networks": len(all_stats),
            "correctness_gate": ("PASSED: for every enumerated s-t path, the composition "
                                 "of gold atomic relations contains the gold query "
                                 "relation (asserted on all networks)."),
            "deduction_required": ("query pair (s,t) shares no edge and never co-occurs in "
                                   "a single verbalized sentence"),
            "reproducible": "per (algebra, cell, index) md5 seeds; resumable",
        },
        "splits": {"scheme": "deterministic md5(seed)%100 within every cell",
                   "pilot": "<10", "dev": "10-29", "test": ">=30",
                   "fold_counts": {a: dict(fc) for a, fc in fold.items()}},
        "relation_histograms": {a: dict(rel_hist[a]) for a in rel_hist},
        "cell_summary": cell_summary,
        "realism_preregistration": REALISM_PREREG,
        "row_field_guide": {
            "input": "NL realization (one sentence per non-query edge + Query line)",
            "output": "JSON string: {edges:[{source,target,relation}], query_edge:{...,is_query}}",
            "metadata_paths": "enumerated s-t paths, per-path gold composition, naive "
                              "intersection, has_bite, singleton_resolved",
            "metadata_structure": "num_nodes/edges, cyclomatic_number, cycle_basis_size, "
                                  "num_simple_paths_s_t, paths_truncated, contributing_edge_count",
            "metadata_reference_disjunctive_labels": "SOUND superset per edge (convenience; "
                                                     "convex-only for point)",
        },
    }

### Worker bridge

`_job` unpacks a job spec and builds one network, failing loudly with context. Copied
verbatim from `method.py`.

In [ ]:
def _job(spec):
    algebra, cell, seed, index = spec
    try:
        return build_network(algebra, cell, seed, index)
    except Exception as e:  # noqa: BLE001 - fail loud with context
        logger.error(f"build failed alg={algebra} cell={cell['cell_id']} seed={seed}: {e!r}")
        raise

## Run the generator (small scale)

This mirrors the original `main()` — build the grid, queue one job per
`(algebra, cell, index)`, run them through the correctness gate, plant any
under-represented relations, and assemble the three datasets + QA metadata. The **two
demo changes** vs. the original: the multiprocessing `Pool` is replaced by a plain serial
loop, and the file-splitting/writing step is dropped (we keep the assembled object in
memory). Scale comes from the config cell.

In [ ]:
def run_demo(networks_primary, networks_secondary, cell_filter=None):
    grid = build_grid()
    jobs = []
    for algebra, cells in grid.items():
        n_per = networks_primary if algebra in PRIMARY else networks_secondary
        for cell in cells:
            if cell_filter is not None and not cell_filter(cell):
                continue
            for i in range(n_per):
                jobs.append((algebra, cell, net_seed(algebra, cell["cell_id"], i), i))
    print(f"[demo] {len(jobs)} networks queued "
          f"(primary={networks_primary}/cell, secondary={networks_secondary}/cell)")

    rows = {alg: [] for alg in grid}
    all_stats = []
    # DEMO CHANGE: original uses a multiprocessing Pool; here we loop serially in-process.
    for spec in jobs:
        row, stats = _job(spec)
        rows[stats["algebra"]].append((stats["cell_id"], stats["fold"], row))
        all_stats.append(stats)
    print(f"[demo] GATE: all {len(all_stats)} networks passed the path-composition "
          f"correctness gate")

    # ---- relation-coverage check + planting fallback (verbatim from main) ----------
    rel_hist = {alg: Counter() for alg in grid}
    for st in all_stats:
        rel_hist[st["algebra"]].update(st["rel_counts"])
    for algebra in grid:
        missing = [r for r in ALG[algebra].base if rel_hist[algebra][r] < 3]
        for r in missing:
            for k in range(5):
                prow = build_coverage_row(algebra, r, net_seed(algebra, f"cov_{r}", k))
                rows[algebra].append((f"coverage_{r}", prow["metadata_fold"], prow))
                rel_hist[algebra].update(
                    Counter([e[2] for e in prow["metadata_abstract_graph"]["edges"]]
                            + [prow["metadata_query"]["relation"]]))
        if missing:
            print(f"[demo] planted under-represented {algebra} relations: {missing}")
        still = [r for r in ALG[algebra].base if rel_hist[algebra][r] == 0]
        assert not still, f"{algebra} relations never realized: {still}"

    # ---- assemble dataset object (verbatim from main) ------------------------------
    datasets = []
    for algebra in ("point", "allen", "rcc8"):
        examples = [row for (_cid, _f, row) in rows[algebra]]
        datasets.append({"dataset": f"synthetic_qcn_{algebra}", "examples": examples})

    args = types.SimpleNamespace(tag="demo", networks_primary=networks_primary,
                                 networks_secondary=networks_secondary)
    qa = build_qa(all_stats, rel_hist, rows, args)
    obj = {"metadata": qa, "datasets": datasets}
    print(f"[demo] assembled: point={len(datasets[0]['examples'])}, "
          f"allen={len(datasets[1]['examples'])}, rcc8={len(datasets[2]['examples'])} examples")
    return obj


gen = run_demo(NETWORKS_PER_CELL_PRIMARY, NETWORKS_PER_CELL_SECONDARY)

## Inspect one generated network

Pick a redundant allen network (`P≥2`) and unpack it the way a consumer would: the NL
document (input), the gold atomic graph (output), the enumerated `s→t` deduction paths
with their per-path gold compositions, and the one-pass `naive_intersection` certificate
(`has_bite` / `singleton_resolved`).

In [ ]:
allen_rows = [r for r in gen["datasets"][1]["examples"]
              if (r["metadata_cell"]["redundancy_P"] or 0) >= 2]
# prefer a fully-resolved one if present (the query is pinned to a single relation)
ex = next((r for r in allen_rows if r["metadata_paths"]["singleton_resolved"]),
          allen_rows[0])

out = json.loads(ex["output"])
q = ex["metadata_query"]
print("cell:", ex["metadata_cell"]["cell_id"],
      "| algebra:", ex["metadata_algebra"], "| fold:", ex["metadata_fold"])
print("\n--- INPUT (NL realization) ---\n" + ex["input"])
print("\n--- OUTPUT: gold atomic graph ---")
for e in out["edges"]:
    print(f"  ({e['source']} -> {e['target']}) : {e['relation']}")
print(f"  QUERY ({q['source']} -> {q['target']}) : {q['relation']}  [held out]")

print("\n--- enumerated s->t deduction paths + gold composition ---")
mp = ex["metadata_paths"]
for path, comp in zip(mp["path_list"], mp["path_compositions"]):
    contains = q["relation"] in comp
    print(f"  path {path}: composition={comp}  (contains gold query '{q['relation']}': {contains})")
print(f"\nnaive_intersection (one-pass certificate): {mp['naive_intersection']}")
print(f"has_bite={mp['has_bite']}  singleton_resolved={mp['singleton_resolved']}")
st = ex["metadata_structure"]
print(f"structure: nodes={st['num_nodes']} edges={st['num_edges']} "
      f"mu={st['cyclomatic_number']} #paths={st['num_simple_paths_s_t']}")

## The headline structural signal: singleton-resolution rises with redundancy `P`

The intended signal is that the one-pass certificate pins the query to a **single**
relation (`singleton_resolved`) more often as redundancy `P` grows — more independent
`s→t` paths give more intersecting constraints. We read the demo-generated
`cell_summary` and overlay the **published full-corpus** values (from the GitHub subset's
metadata) on the redundancy sweep (`allen`, `L=2`).

In [ ]:
def redundancy_curve(cell_summary, algebra="allen"):
    """Return ([P...], [singleton_frac...]) for the red_P{P}_L2 cells, sorted by P."""
    pts = []
    for key, summ in cell_summary.items():
        alg, cid = key.split("/")
        if alg == algebra and cid.startswith("red_P") and cid.endswith("_L2"):
            P = int(cid[len("red_P"):cid.index("_L2")])
            pts.append((P, summ["singleton_resolved_frac"], summ["count"]))
    pts.sort()
    return [p for p, _, _ in pts], [f for _, f, _ in pts], [c for _, _, c in pts]

gen_P, gen_f, gen_c = redundancy_curve(gen["metadata"]["cell_summary"])
pub_P, pub_f, pub_c = redundancy_curve(data["metadata"]["cell_summary"])

print("redundancy P :", gen_P)
print("demo singleton-resolved frac :", [round(x, 3) for x in gen_f],
      "(", gen_c, "networks/cell )")
print("full-corpus singleton-resolved frac :", [round(x, 3) for x in pub_f])

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(gen_P, gen_f, "o-", label=f"demo ({gen_c[0]} nets/cell)", lw=2)
if pub_P:
    ax.plot(pub_P, pub_f, "s--", color="gray", label="published full corpus", lw=2)
ax.set_xlabel("redundancy  P  (number of independent s->t paths, L=2)")
ax.set_ylabel("singleton_resolved fraction")
ax.set_title("Allen: query resolves to a singleton more often as redundancy P rises")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Demo summary

A compact view of the generated corpus: per-algebra example counts, the overall
correctness-gate result, and a per-cell summary table for the allen redundancy sweep.
Finally, one published row from the loaded subset confirms the demo output matches the
canonical published schema.

In [ ]:
print("=== generated corpus (demo scale) ===")
for d in gen["datasets"]:
    n = len(d["examples"])
    bite = sum(r["metadata_paths"]["has_bite"] for r in d["examples"])
    sing = sum(r["metadata_paths"]["singleton_resolved"] for r in d["examples"])
    print(f"  {d['dataset']:22s}: {n:4d} rows | has_bite {bite:4d} | singleton {sing:4d}")
print(f"\ncorrectness gate: {gen['metadata']['generation']['correctness_gate']}")

print("\n=== allen redundancy sweep (demo) ===")
print(f"{'cell':12s} {'count':>6s} {'has_bite':>9s} {'singleton':>10s} {'mean_paths':>11s}")
cs = gen["metadata"]["cell_summary"]
for key in sorted(cs):
    alg, cid = key.split("/")
    if alg == "allen" and cid.startswith("red_P") and cid.endswith("_L2"):
        s = cs[key]
        print(f"{cid:12s} {s['count']:6d} {s['has_bite_frac']:9.3f} "
              f"{s['singleton_resolved_frac']:10.3f} {s['mean_num_simple_paths']:11.3f}")

print("\n=== one PUBLISHED row from the loaded GitHub subset (canonical schema) ===")
pub = data["datasets"][0]["examples"][0]
print("input :", pub["input"].replace("\n", " "))
print("output:", pub["output"])
print("cell  :", pub["metadata_cell"]["cell_id"], "| query:", pub["metadata_query"])